# Unweighted sampling laws

Inspect the four 500-secant sampling estimates and export their distributions and compatibility values. Absolute values use the unitary-FFT convention.


In [ ]:
from pathlib import Path
import sys

import importlib
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_candidates = [
        candidate / 'sd15_conditioning_experiment.py',
        candidate / 'analyze_results' / 'sd15_conditioning_experiment.py',
        candidate / 'sd1.5' / 'analyze_results' / 'sd15_conditioning_experiment.py',
    ]
    for helper_path in helper_candidates:
        if helper_path.is_file():
            helper_dir = helper_path.parent
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_conditioning_experiment.py from the notebook cwd.')

import sd15_conditioning_experiment as exp
exp = importlib.reload(exp)

SD15_ROOT = exp.find_sd15_root(NOTEBOOK_DIR)
CATALOG = exp.load_ktilde_catalog(SD15_ROOT)
TABLES = exp.build_lambda_tables(SD15_ROOT, skip_missing=True)
BANK_SUMMARY = pd.DataFrame(
    [
        {
            'name': name,
            'role': value.get('role', name),
            'label': value.get('label', name),
            'christoffel_law': value.get('christoffel_law', ''),
            'prompt': value.get('prompt', ''),
            'prompt_bank': ', '.join(value.get('prompt_bank', [])),
            'artifact_exists': (SD15_ROOT / 'unweighted' / 'ktilde' / f'{name}.npz').is_file(),
        }
        for name, value in CATALOG.items()
    ]
)


def style_plain_numbers(frame):
    return frame.style.format(lambda value: exp.format_plain_number(value))


display(BANK_SUMMARY)
if TABLES['missing_names']:
    print('Missing k-tilde artifacts:', TABLES['missing_names'])
FFT_ENERGY_SCALES = sorted({int(round(value)) for value in TABLES['fft_energy_scale'].values()})
display(
    Markdown(
        "**FFT normalization.** "
        "Stored `K_tilde` artifacts were estimated with the unnormalized FFT, so the absolute lambda and kappa tables below divide Fourier energies by `H * W` "
        f"({', '.join(str(value) for value in FFT_ENERGY_SCALES)}) to report the unitary-FFT convention. "
        "The mismatch table is unchanged because this scale cancels."
    )
)
display(TABLES['kappa_df'].style.format({'kappa_hat': exp.format_plain_number}))
display(
    TABLES['lambda_df'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'mismatch_penalty': exp.format_plain_number,
        }
    )
)
display(style_plain_numbers(TABLES['lambda_table']))
display(style_plain_numbers(TABLES['penalty_table']))
display(
    TABLES['matched_check'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'abs_lambda_minus_kappa': exp.format_plain_number,
        }
    )
)
display(
    Markdown(
        r"**$\widetilde{\Lambda}'$ interpretation.** "
        r"The $\widetilde{\lambda}$ heatmap shows the absolute compatibility cost of using sampling law "
        r"$\widetilde{\mu}_{c_s}$ with Christoffel function $c_r$. "
        r"$\widetilde{\Lambda}'$ is the row-normalized mismatch factor "
        r"$\widetilde{\lambda}(c_r,c_r,c_s)/\widetilde{\kappa}(c_r)$, so values near 1 mean the sampling law is close to the matched baseline for that row, while larger values show how much extra penalty you pay from mismatch."
    )
)


## Export Lambda Figures

This cell exports the absolute Lambda heatmap, the individual sampling-law plots, and the compact sampling-law row under `unweighted/results/ktilde/figures`.

The displayed `FIGURE_PATHS` series is the checklist of files produced by the helper, so it is the quickest way to confirm where the notebook saved each figure.

In [ ]:
FIGURE_DIR = SD15_ROOT / 'unweighted' / 'results' / 'ktilde' / 'figures'
FIGURE_PATHS = exp.export_lambda_figure_set(
    TABLES,
    output_dir=FIGURE_DIR,
    file_format='pdf',
    show=True,
)
display(pd.Series({key: str(value) for key, value in FIGURE_PATHS.items()}))
FIGURE_PATHS
